In [ ]:
import zipfile

BASE = "/content/drive/MyDrive/traffic_app/dataset"

zip_path = f"{BASE}/images.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(f"{BASE}/raw_data")

print("Extraction ho gayi. Files yahan hain:", f"{BASE}/raw_data")


import os
files = os.listdir(f"{BASE}/raw_data")
print("Total files:", len(files))
print("Sample files:", files[:5])

Extraction ho gayi. Files yahan hain: /content/drive/MyDrive/traffic_app/dataset/raw_data
Total files: 3
Sample files: ['images', 'labels', 'data.yaml']


In [ ]:
import os

BASE = "/content/drive/MyDrive/traffic_app/dataset"

for split in ["train", "val", "test"]:
    img_count = len(os.listdir(f"{BASE}/raw_data/images/{split}"))
    lbl_count = len(os.listdir(f"{BASE}/raw_data/labels/{split}"))
    print(f"{split}: {img_count} images, {lbl_count} labels")

train: 143 images, 143 labels
val: 38 images, 38 labels
test: 21 images, 21 labels


In [ ]:
PROJECT = "/content/drive/MyDrive/traffic_app"

data_yaml_content = """path: /content/drive/MyDrive/traffic_app/dataset
train: images/train
val: images/val
test: images/test

names:
  0: bus
  1: car
  2: motorcycle
  3: person
  4: truck
"""

with open(f"{PROJECT}/dataset/data.yaml", "w") as f:
    f.write(data_yaml_content)

print("data.yaml corrected and saved")
print(data_yaml_content)

data.yaml corrected and saved
path: /content/drive/MyDrive/traffic_app/dataset
train: images/train
val: images/val
test: images/test

names:
  0: bus
  1: car
  2: motorcycle
  3: person
  4: truck



In [ ]:
!pip install -U ultralytics -q

import torch
print("GPU Available:", torch.cuda.is_available())
!nvidia-smi

GPU Available: True
Sun Aug  9 03:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [ ]:
from ultralytics import YOLO
import time
import os

model = YOLO("yolov8n.pt")


corrected_data_yaml_content = f"""path: {PROJECT}/dataset
train: raw_data/images/train
val: raw_data/images/val
test: raw_data/images/test

names:
  0: bus
  1: car
  2: motorcycle
  3: person
  4: truck
"""


corrected_data_yaml_path = f"{PROJECT}/dataset/data.yaml"
with open(corrected_data_yaml_path, "w") as f:
    f.write(corrected_data_yaml_content)

print("Corrected data.yaml saved at:", corrected_data_yaml_path)

start_time = time.time()

results = model.train(
    data=corrected_data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=8,
    pretrained=True,
    project=f"{PROJECT}/outputs/training_results",
    name="yolov8n_traffic_fixed",
)

print(f"Training completed in {(time.time()-start_time)/60:.2f} minutes")

Corrected data.yaml saved at: /content/drive/MyDrive/traffic_app/dataset/data.yaml
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/traffic_app/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tr

In [ ]:
import shutil

src = f"{PROJECT}/outputs/training_results/yolov8n_traffic_fixed-2/weights/best.pt"
dst = f"{PROJECT}/models/best.pt"
shutil.copy(src, dst)
print("Corrected best.pt saved to:", dst)

Corrected best.pt saved to: /content/drive/MyDrive/traffic_app/models/best.pt


In [ ]:
metrics = model.val(
    data=f"{BASE}/data.yaml",
    project=f"{PROJECT}/outputs/training_results",
    name="yolov8n_traffic_val",
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.4±0.8 ms, read: 54.7±29.2 MB/s, size: 55.6 KB)
val: Scanning /content/drive/MyDrive/traffic_app/dataset/raw_data/labels/val.cache... 38 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 10.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0s/it 3.1s
                   all         38         93      0.476      0.675      0.631      0.386
                   bus          9         10      0.499        0.8      0.727      0.518
                   car         27         56      0.729      0.857      0.868      0.638
            motorcycle          7          7      0.357      0.857      0.797      0.398
                person          5         17       0.49      0.529      0.533      0.197
                 truck 

In [ ]:
predict_model = YOLO(dst)

predict_results = predict_model.predict(
    source=f"{BASE}/raw_data/images/test",
    conf=0.35,
    save=True,
    project=f"{PROJECT}/outputs/predictions",
    name="test_predictions",
)

print("Predictions saved. Check outputs/predictions/test_predictions folder.")


image 1/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/101_jpg.rf.1d96eba3aeafa61970f4e015389c976f.jpg: 640x640 3 cars, 60.5ms
image 2/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/102_jpg.rf.65c42b81b3864eceb0c7e3543fe36626.jpg: 640x640 4 cars, 80.4ms
image 3/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/118_jpg.rf.2a142c9ea492f7e605fd65e8b96f269a.jpg: 640x640 5 cars, 70.8ms
image 4/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/123_jpg.rf.dca462f30045e87f16dd5b5b09aa60eb.jpg: 640x640 4 cars, 51.2ms
image 5/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/456_jpg.rf.e210bd0c48932a365c8eb3ad608d136d.jpg: 640x640 2 cars, 1 motorcycle, 24.8ms
image 6/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/56_jpg.rf.c000752863edca07a1457141090d1db9.jpg: 640x640 1 bus, 5 cars, 57.7ms
image 7/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/57_jpg.rf.b8bf2071cae5

In [ ]:
label_dir = f"{PROJECT}/dataset/raw_data/labels/train"
sample_label = os.listdir(label_dir)[0]

with open(f"{label_dir}/{sample_label}") as f:
    print("Sample label file:", sample_label)
    print(f.read())

Sample label file: 10_jpg.rf.b993fb558c207754b79840928384e821.txt
1 0.0203125 0.14453125 0.040625 0.0609375
1 0.146875 0.2140625 0.0671875 0.10859375
1 0.08046875 0.2859375 0.05234375 0.08359375
1 0.0234375 0.315625 0.0390625 0.05703125
1 0.13515625 0.034375 0.06640625 0.06796875
1 0.26875 0.03671875 0.09453125 0.0734375
1 0.06171875 0.421875 0.0484375 0.0890625
1 0.09765625 0.5359375 0.059375 0.103125
1 0.0875 0.06015625 0.05625 0.1203125
1 0.05546875 0.096875 0.03125 0.08203125


In [ ]:
raw_yaml_path = f"{PROJECT}/dataset/raw_data/data.yaml"

with open(raw_yaml_path) as f:
    print(f.read())

# =========================================================
# YOLOv8 Dataset Configuration — Traffic Object Detection
# Source: Roboflow Universe (vehicle-detection-hkjb5-s9a0n, v1, CC BY 4.0)
# =========================================================
path: /content/drive/MyDrive/cv_project/dataset
train: images/train
val: images/val
test: images/test

# Class order kept exactly as exported from Roboflow — do not renumber.
# The numbers in every label .txt file already correspond to this order.
names:
  0: bus
  1: car
  2: motorcycle
  3: person
  4: truck



test
# New Section

In [ ]:
import os

pred_root = f"{PROJECT}/outputs/predictions"
print(os.listdir(pred_root))

['test_predictions', 'test_predictions-2', 'test_predictions-3', 'test_predictions-4', 'test_predictions-5', 'test_predictions-6', 'test_predictions-7', 'test_predictions-8', 'test_predictions-9', 'test_predictions-10', 'test_predictions_FINAL_FIXED', 'test_predictions-11', 'test_predictions_FINAL_FIXED-2', 'test_predictions-12', 'test_predictions_FINAL_FIXED-3', 'test_predictions-13', 'test_predictions_FINAL_FIXED-4', 'test_predictions-14', 'test_predictions_FINAL_FIXED-5', 'test_predictions-15']


In [ ]:
import os
model_path = f"{PROJECT}/models/best.pt"
print("Last modified:", os.path.getmtime(model_path))

import datetime
print("Modified on:", datetime.datetime.fromtimestamp(os.path.getmtime(model_path)))

In [ ]:
predict_model = YOLO(f"{PROJECT}/models/best.pt")

predict_results = predict_model.predict(
    source=f"{PROJECT}/dataset/raw_data/images/test",
    conf=0.35,
    save=True,
    project=f"{PROJECT}/outputs/predictions",
    name="test_predictions_FINAL_FIXED",
)


image 1/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/101_jpg.rf.1d96eba3aeafa61970f4e015389c976f.jpg: 640x640 3 cars, 41.5ms
image 2/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/102_jpg.rf.65c42b81b3864eceb0c7e3543fe36626.jpg: 640x640 4 cars, 31.9ms
image 3/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/118_jpg.rf.2a142c9ea492f7e605fd65e8b96f269a.jpg: 640x640 5 cars, 26.3ms
image 4/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/123_jpg.rf.dca462f30045e87f16dd5b5b09aa60eb.jpg: 640x640 4 cars, 50.2ms
image 5/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/456_jpg.rf.e210bd0c48932a365c8eb3ad608d136d.jpg: 640x640 2 cars, 1 motorcycle, 24.6ms
image 6/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/56_jpg.rf.c000752863edca07a1457141090d1db9.jpg: 640x640 1 bus, 5 cars, 24.0ms
image 7/21 /content/drive/MyDrive/traffic_app/dataset/raw_data/images/test/57_jpg.rf.b8bf2071cae5

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

pred_dir = f"{PROJECT}/outputs/predictions/test_predictions-10"

pred_images = os.listdir(pred_dir)[:15]

fig, axes = plt.subplots(3, 5, figsize=(25, 15))
for ax, img_name in zip(axes.flatten(), pred_images):
    img = mpimg.imread(f"{pred_dir}/{img_name}")
    ax.imshow(img)
    ax.set_title(img_name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os

for cls in ['bus', 'truck']:
    count = 0
    for split in ['train', 'val']:
        label_dir = f"{PROJECT}/dataset/raw_data/labels/{split}"
        for lbl_file in os.listdir(label_dir):
            with open(f"{label_dir}/{lbl_file}") as f:
                for line in f:
                    if int(line.split()[0]) == (0 if cls=='bus' else 4):
                        count += 1
    print(f"{cls}: {count} instances")

bus: 46 instances
truck: 11 instances


# FastApi integration

In [ ]:
%%writefile /content/drive/MyDrive/traffic_app/app/app.py
import io
import base64
import numpy as np
from PIL import Image
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/traffic_app/models/best.pt"
CONF_THRESHOLD = 0.35

app = FastAPI(title="Traffic Object Detection API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

model = YOLO(MODEL_PATH)

@app.get("/")
def root():
    return {"message": "Traffic Detection API running. POST an image to /predict."}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    results = model.predict(source=np.array(image), conf=CONF_THRESHOLD, verbose=False)
    result = results[0]

    detections = []
    for box in result.boxes:
        cls_id = int(box.cls[0])
        detections.append({
            "class_name": model.names[cls_id],
            "confidence": round(float(box.conf[0]), 4),
            "bbox_xyxy": [round(float(x), 2) for x in box.xyxy[0].tolist()],
        })

    annotated = result.plot()
    annotated_img = Image.fromarray(annotated[:, :, ::-1])
    buf = io.BytesIO()
    annotated_img.save(buf, format="JPEG")
    annotated_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    return JSONResponse({
        "num_detections": len(detections),
        "detections": detections,
        "annotated_image_base64": annotated_b64,
    })

Overwriting /content/drive/MyDrive/traffic_app/app/app.py


In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok nest_asyncio -q

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import sys
import asyncio

sys.path.append("/content/drive/MyDrive/traffic_app/app")
from app import app

nest_asyncio.apply()


ngrok.set_auth_token("3HTxEyL6iKaCJBNvbbWygALxg3O_72XyCE7Vtx1oQXjKzHx77")


public_url = ngrok.connect(8000)
print("API live at:", public_url)
print("Test karne ke liye kholein:", str(public_url) + "/docs")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)


async def run_server():
    await server.serve()


loop = asyncio.get_event_loop()
if loop.is_running():
    loop.create_task(run_server())
else:
    loop.run_until_complete(run_server())

API live at: NgrokTunnel: "https://stoop-banshee-overeager.ngrok-free.dev" -> "http://localhost:8000"
Test karne ke liye kholein: NgrokTunnel: "https://stoop-banshee-overeager.ngrok-free.dev" -> "http://localhost:8000"/docs


In [ ]:
import pandas as pd
import os
from ultralytics import YOLO

class_names = {0: "bus", 1: "car", 2: "motorcycle", 3: "person", 4: "truck"}
model = YOLO(f"{PROJECT}/models/best.pt")

test_img_dir = f"{PROJECT}/dataset/raw_data/images/test"
test_lbl_dir = f"{PROJECT}/dataset/raw_data/labels/test"

error_rows = []

for img_file in os.listdir(test_img_dir):
    img_path = f"{test_img_dir}/{img_file}"
    label_file = os.path.splitext(img_file)[0] + ".txt"
    label_path = f"{test_lbl_dir}/{label_file}"

    if not os.path.exists(label_path):
        continue

    with open(label_path) as f:
        actual_classes = set(class_names[int(line.split()[0])] for line in f if line.strip())

    results = model.predict(source=img_path, conf=0.35, verbose=False)
    predicted_classes = set(class_names[int(box.cls[0])] for box in results[0].boxes)

    if actual_classes == predicted_classes:
        continue

    missing = actual_classes - predicted_classes
    extra = predicted_classes - actual_classes


    if missing and extra:
        error_type = "Wrong class"
        reason = f"Model saw '{', '.join(extra)}' instead of '{', '.join(missing)}'"
    elif missing and not extra:
        error_type = "False negative"
        reason = f"Missed detecting: {', '.join(missing)}"
    elif extra and not missing:
        error_type = "False positive"
        reason = f"Detected extra object(s) not in ground truth: {', '.join(extra)}"
    else:
        error_type = "Unknown"
        reason = "review manually"

    error_rows.append({
        "image": img_file,
        "actual_object": ", ".join(sorted(actual_classes)) if actual_classes else "none",
        "model_prediction": ", ".join(sorted(predicted_classes)) if predicted_classes else "not detected",
        "error_type": error_type,
        "possible_reason": reason
    })

df = pd.DataFrame(error_rows)
print(f"Total mismatches found: {len(df)}")
df.to_csv(f"{PROJECT}/report/error_analysis_auto.csv", index=False)
df

Total mismatches found: 5


,image,actual_object,model_prediction,error_type,possible_reason
0,57_jpg.rf.b8bf2071cae5ca9e5a983a97b46e5418.jpg,"bus, car",car,False negative,Missed detecting: bus
1,img355_jpg.rf.9d1bdb1361833ce533d43e00264063ba...,bus,"bus, car, person",False positive,Detected extra object(s) not in ground truth: ...
2,img180_jpg.rf.4188d271081f2968face8092d23b1ea0...,truck,bus,Wrong class,Model saw 'bus' instead of 'truck'
3,img643_jpg.rf.4f4e6b5f10950509de5f1a25c938c24b...,"bus, person",bus,False negative,Missed detecting: person
4,img639_jpg.rf.151ab99669ca1b67d90f40fc1c339409...,"bus, person",bus,False negative,Missed detecting: person


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1WSzya7_tirBe5ZW_DG5pRwBsSNXPETgYgBodqWXyAg0/edit#gid=0


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/traffic_app/app")

if 'app' in sys.modules:
    del sys.modules['app']

from app import app

middleware_names = [m.cls.__name__ for m in app.user_middleware]
print("Active middleware:", middleware_names)

Active middleware: ['CORSMiddleware']


In [ ]:
with open("/content/drive/MyDrive/traffic_app/app/app.py") as f:
    content = f.read()
print(content)

import io
import base64
import numpy as np
from PIL import Image
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/traffic_app/models/best.pt"
CONF_THRESHOLD = 0.35

app = FastAPI(title="Traffic Object Detection API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

model = YOLO(MODEL_PATH)

@app.get("/")
def root():
    return {"message": "Traffic Detection API running. POST an image to /predict."}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    results = model.predict(source=np.array(image), conf=CONF_THRESHOLD, verbose=False)
    result = results[0]

    detections = []
    for box in result.boxes:
        cls_id = int(box.cls[0])
    

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/traffic_app

/content/drive/MyDrive/traffic_app


In [4]:
!find /content/drive/MyDrive -maxdepth 3 -iname "traffic_app*"

/content/drive/MyDrive/traffic_app


In [5]:
!git config --global user.name "mehreen18"
!git config --global user.email "mehreenm388@gmail.com"

In [6]:
!git status

fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [7]:
!git init
!git remote add origin https://github.com/mehreen18/traffic-detection-yolov8.git

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/traffic_app/.git/


In [8]:
!git branch -m main

In [9]:
!git remote add origin https://github.com/mehreen18/traffic-detection-yolov8.git
!git remote -v

error: remote origin already exists.
origin	https://github.com/mehreen18/traffic-detection-yolov8.git (fetch)
origin	https://github.com/mehreen18/traffic-detection-yolov8.git (push)


In [10]:
!git config --global user.name "mehreen18"
!git config --global user.email "mehreenm388@gmail.com"

In [11]:
!git add .
!git commit -m "Initial push of traffic_app project from Colab"

^C
On branch main

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md
	app/
	dataset/
	models/
	notebooks/
	outputs/
	report/
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)


In [12]:
!git reset


In [15]:
%%writefile .gitignore
# Dataset
dataset/

# Outputs
outputs/

# Cache
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [16]:
!du -sh * | sort -rh | head -20

439M	outputs
22M	dataset
6.0M	models
81K	notebooks
14K	app
6.5K	report
1.0K	README.md
512	requirements.txt


In [18]:
!git commit -m "Initial push of traffic_app project from Colab"

On branch main

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	app/
	models/
	notebooks/
	report/
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)
